# 🏢 LeaveFlow AI — LangGraph Leave Management System
### Production-Grade Multi-Agent System with Full Step-by-Step Visibility

---

## 📌 What This Notebook Teaches You

This is a **complete, runnable LangGraph project** designed to be your primary reference for building multi-agent AI systems at an MNC level. Every concept is explained in plain language, and **every node prints its full input and output** so you can see exactly what's happening at each step.

---

## 🧠 Before We Code — Core Theory You MUST Understand

### What is LangGraph?
LangGraph is a library built on top of LangChain that lets you build **stateful, multi-step AI applications** structured as a **graph**. Think of it like this:

- **Nodes** = Functions (each does one specific job, like a specialist in a team)
- **Edges** = Paths between nodes (like roads between cities)
- **State** = A shared dictionary passed through every node (like a file folder that everyone reads and writes to)
- **Conditional Edges** = Smart routing — "if X happened, go to node A; else go to node B"

### Why not just chain LLM calls in sequence?
Because real workflows are **not linear**. Sometimes you need to:
- Loop back (ask user for missing info again)
- Branch (skip KT if leave < 3 days)
- Pause and wait (for manager approval)
- Handle errors differently per step

LangGraph handles ALL of this elegantly.

### The Mental Model for This Project
```
User says → "I want 3 days CL from Monday"
         ↓
  [N1] Intent Parser      → extracts: leave_type=CL, num_days=3, start=Monday
         ↓
  [N2] Pattern Analyser   → checks history: any monday pattern?
         ↓
  [N3] RAG Policy Reader  → reads policy: CL rules, max days, restrictions
         ↓
  [N4] Eligibility Check  → balance check: do they have 3 CL days?
         ↓ (if eligible)
  [N5] Leave Optimizer    → generates 4 plan options → user picks one
         ↓ (3 days, so KT required)
  [N6] KT Validator       → collects knowledge transfer details
         ↓
  [N7] Notification Agent → emails manager, PAUSES for approval
         ↓ (if approved)
  [N8] Transaction Exec   → saves to DB, deducts balance, sends confirmation
```

---

## 📂 Notebook Structure

| Section | What You'll Learn |
|---------|-------------------|
| **Cell 1** | Setup & Imports — what each library does |
| **Cell 2** | State Design — the heart of LangGraph |
| **Cell 3** | Mock Data Layer — simulating databases |
| **Cell 4** | Utility Functions — helper tools |
| **Cell 5** | Node 1: Intent Parser |
| **Cell 6** | Node 2: Pattern Analyser |
| **Cell 7** | Node 3: RAG Policy Reader |
| **Cell 8** | Node 4: Eligibility Checker |
| **Cell 9** | Node 5: Leave Plan Optimizer |
| **Cell 10** | Node 6: KT Validator |
| **Cell 11** | Node 7: Notification Agent |
| **Cell 12** | Node 8: Transaction Executor |
| **Cell 13** | Routing Functions — all conditional edges |
| **Cell 14** | Graph Assembly — wiring everything together |
| **Cell 15** | 🚀 Full Run — Happy Path (eligible, approved) |
| **Cell 16** | 🔴 Edge Case Run — Ineligible employee |
| **Cell 17** | 🔴 Edge Case Run — Short leave (no KT needed) |
| **Cell 18** | 🔴 Edge Case Run — Pattern detected |
| **Cell 19** | Graph Visualization |
| **Cell 20** | Production Checklist & What to Build Next |

---
## 📦 CELL 1 — Setup & Imports

### Theory: What Each Library Does

| Library | Role in This Project |
|---------|---------------------|
| `langgraph` | The graph engine — builds nodes, edges, state machine |
| `langchain_openai` | Connects to OpenAI's GPT models |
| `langchain_core` | Base types: messages, prompts, output parsers |
| `typing` | Python type hints — makes state types explicit and safe |
| `datetime` | Date calculations for leave duration, weekday detection |
| `json` | Parsing LLM outputs (we prompt the LLM to return JSON) |
| `os` | Reading environment variables (API key) |

> **Why TypedDict for State?**  
> TypedDict gives us a dictionary with **declared field types**. LangGraph uses this to:
> 1. Know what fields exist in the state
> 2. Validate that nodes return the right types
> 3. Merge partial updates (each node only returns the fields it changed)

In [ ]:
# ============================================================
# CELL 1: Setup & Imports
# Run this first — always.
# ============================================================

import os
import json
import re
from datetime import datetime, timedelta, date
from typing import TypedDict, List, Optional, Dict, Any, Annotated
from enum import Enum

# LangGraph core
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# LangChain OpenAI integration
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

# ──────────────────────────────────────────────────
# SET YOUR API KEY HERE
# Option 1: Set directly (for local/Jupyter use)
# Option 2: Use environment variable (recommended for production)
# ──────────────────────────────────────────────────
os.environ["OPENAI_API_KEY"] = "sk-your-openai-api-key-here"  # ← REPLACE THIS

# Initialize the LLM
# temperature=0 means deterministic outputs — important for structured data extraction
# temperature>0 is better for creative tasks like generating plan recommendations
llm = ChatOpenAI(
    model="gpt-4o-mini",     # Cost-effective, fast, good enough for structured extraction
    temperature=0,           # Zero randomness for parsing/extraction nodes
    max_tokens=2000,
)

llm_creative = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,         # Slight creativity for plan generation
    max_tokens=2000,
)

print("✅ All imports successful")
print(f"✅ LLM initialized: {llm.model_name}")
print("✅ Ready to build the graph")
print()
print("📌 IMPORTANT: Replace 'sk-your-openai-api-key-here' with your actual key before running further cells")

---
## 🗂️ CELL 2 — State Design

### Theory: The State is EVERYTHING in LangGraph

**The state is the single source of truth.** Every node reads from it, every node writes to it. Think of it as the "application form" being passed through a government office — each desk fills in their section.

### Key LangGraph State Rules:
1. **State is a TypedDict** — a dictionary with named, typed fields
2. **Nodes don't communicate with each other** — they only read/write state
3. **Nodes return only the fields they changed** — LangGraph merges the rest
4. **The router reads state** — routing decisions are based on state values

### State Lifecycle Example:
```
Initial state:  { raw_input: "I want 3 CL", emp_id: "", leave_type: "", ... }
After N1:       { raw_input: "I want 3 CL", emp_id: "EMP-001", leave_type: "CL", num_days: 3, ... }
After N2:       { ...same + pattern_flags: [], pattern_risk: "low" }
After N3:       { ...same + policy_summary: "...", policy_rules: {...} }
After N4:       { ...same + balance_CL: 5, is_eligible: True }
...and so on
```

### What is `Annotated[list, add_messages]`?
This is a special LangGraph type for the `messages` field. Instead of replacing the list on each update, it **appends** new messages. This is how conversation history is maintained across nodes.

In [ ]:
# ============================================================
# CELL 2: State Design — The Shared State Object
# ============================================================

class LeaveApplicationState(TypedDict):
    """
    THE CENTRAL STATE OBJECT.
    
    Every node receives the full state and returns a dict
    with only the fields it updated. LangGraph auto-merges.
    
    Fields are grouped by which node primarily sets them.
    """

    # ── CONVERSATION ────────────────────────────────────────
    # Annotated with add_messages = messages are APPENDED, not replaced
    # This preserves full conversation history across all nodes
    messages: Annotated[list, add_messages]

    # ── RAW INPUT ───────────────────────────────────────────
    raw_input: str                    # The employee's original message
    current_node: str                 # Tracks which node is running (for logging)
    errors: List[str]                 # Any errors encountered across nodes
    status: str                       # "in_progress" | "complete" | "rejected" | "error"

    # ── IDENTITY (set by N1: Intent Parser) ─────────────────
    emp_id: str                       # e.g., "EMP-1042"
    emp_name: str                     # e.g., "Priya Sharma"
    emp_email: str                    # e.g., "priya@company.com"
    manager_id: str                   # e.g., "EMP-0010"
    manager_email: str                # e.g., "manager@company.com"
    manager_name: str                 # e.g., "Rajesh Kumar"

    # ── PARSED INTENT (set by N1: Intent Parser) ────────────
    leave_type: str                   # "CL" | "PL" | "SL"
    start_date: str                   # "YYYY-MM-DD" format
    end_date: str                     # "YYYY-MM-DD" format
    num_days: int                     # Total working days requested
    reason: str                       # Optional reason from employee
    missing_info: List[str]           # Fields still needed: ["start_date", "end_date"]
    clarification_question: str       # Question to ask user if missing_info not empty

    # ── PATTERN ANALYSIS (set by N2: Pattern Analyser) ──────
    pattern_flags: List[str]          # e.g., ["monday_pattern", "pre_holiday"]
    pattern_risk: str                 # "low" | "medium" | "high"
    history_summary: str              # Human-readable summary of leave history

    # ── POLICY (set by N3: RAG Policy Reader) ───────────────
    policy_text: str                  # Raw policy text retrieved
    policy_summary: str               # LLM-condensed policy summary
    policy_rules: Dict[str, Any]      # Structured: {max_days, notice_days, restrictions}

    # ── ELIGIBILITY (set by N4: Eligibility Checker) ────────
    balance_CL: int                   # Remaining Casual Leave days
    balance_PL: int                   # Remaining Privilege Leave days
    balance_SL: int                   # Remaining Sick Leave days
    is_eligible: bool                 # Final eligibility verdict
    ineligibility_reason: str         # Why ineligible (if applicable)

    # ── OPTIMIZER (set by N5: Leave Plan Optimizer) ──────────
    leave_plan_options: List[Dict]    # 3-5 plan cards generated by LLM
    selected_plan: Dict               # The plan the user picks
    plan_selected: bool               # True once user has chosen

    # ── KT (set by N6: KT Validator) ────────────────────────
    kt_required: bool                 # True if num_days >= 3
    kt_receiver_id: str               # Employee ID of KT receiver
    kt_receiver_name: str             # Looked up from employee DB
    kt_content: str                   # Minimum 50 words of KT detail
    kt_valid: bool                    # True if KT passes all validations

    # ── NOTIFICATION (set by N7: Notification Agent) ─────────
    notification_sent: bool           # True once email is dispatched
    approval_status: str              # "pending" | "approved" | "rejected"
    email_content: str                # The full email text that was sent

    # ── TRANSACTION (set by N8: Transaction Executor) ────────
    final_record_id: str              # DB record ID after successful save
    confirmation_sent: bool           # True once employee gets confirmation


# ──────────────────────────────────────────────────────────────
# HELPER: Pretty-print the state at any point
# This is how we see "full visibility" of what's in the state
# ──────────────────────────────────────────────────────────────
def print_state(state: dict, section: str = "CURRENT STATE"):
    """
    Prints the current state in a readable format.
    Skips empty/None/[] values to reduce noise.
    Call this inside any node to see what it received.
    """
    print(f"\n{'='*60}")
    print(f"  📋 {section}")
    print(f"{'='*60}")
    
    skip_empty = True
    for key, value in state.items():
        if key == "messages":
            print(f"  messages: [{len(value)} message(s)]")
            for i, msg in enumerate(value[-3:]):  # Show last 3 messages
                role = type(msg).__name__
                content = str(msg.content)[:100] + "..." if len(str(msg.content)) > 100 else str(msg.content)
                print(f"    [{i}] {role}: {content}")
            continue
        if skip_empty and (value is None or value == "" or value == [] or value == {}):
            continue
        # Truncate long values
        display_val = str(value)
        if len(display_val) > 120:
            display_val = display_val[:120] + "..."
        print(f"  {key}: {display_val}")
    print(f"{'='*60}\n")


print("✅ State class defined")
print("✅ print_state() helper ready")
print()
print("📌 Key insight: State has", len(LeaveApplicationState.__annotations__), "fields")
print("📌 Each node only updates the fields it's responsible for")
print("📌 All other fields pass through unchanged — LangGraph handles the merge")

---
## 🗄️ CELL 3 — Mock Data Layer (Simulating Databases)

### Theory: Why Mock Databases?

In a production system, nodes would call real APIs:
- HRMS API for leave balances
- Employee directory API for name/email lookup  
- Oracle DB for leave history
- Vector database for policy search

For this notebook, we simulate all of these with **Python dictionaries and functions**. The node code is identical — only the data source changes. This is exactly how you'd build it: write the nodes against an interface, then swap mock for real later.

### What We're Simulating:
1. **Employee Database** — name, email, manager, department
2. **Leave Balance Database** — how many CL/PL/SL days remain
3. **Leave History Database** — past leave records (for pattern detection)
4. **Policy Store** — the text of the leave policy document
5. **Leave Records DB** — where approved leaves get saved

In [ ]:
# ============================================================
# CELL 3: Mock Data Layer — Simulating Real Databases
# ============================================================

# ── EMPLOYEE DATABASE ────────────────────────────────────────
# In production: HRMS API call or SQL query
EMPLOYEE_DB = {
    "EMP-1042": {
        "name": "Priya Sharma",
        "email": "priya.sharma@company.com",
        "manager_id": "EMP-0010",
        "department": "Engineering",
        "designation": "Senior Developer",
        "probation": False,
        "joining_date": "2022-03-15"
    },
    "EMP-2087": {
        "name": "Amit Verma",
        "email": "amit.verma@company.com",
        "manager_id": "EMP-0010",
        "department": "Engineering",
        "designation": "Developer",
        "probation": False,
        "joining_date": "2023-01-10"
    },
    "EMP-0010": {
        "name": "Rajesh Kumar",
        "email": "rajesh.kumar@company.com",
        "manager_id": "EMP-0001",
        "department": "Engineering",
        "designation": "Engineering Manager",
        "probation": False,
        "joining_date": "2019-06-01"
    },
    # Edge case: employee with no balance
    "EMP-9999": {
        "name": "Test Employee",
        "email": "test@company.com",
        "manager_id": "EMP-0010",
        "department": "Testing",
        "designation": "QA",
        "probation": True,           # On probation — restricted leaves
        "joining_date": "2025-01-01"
    }
}

# ── LEAVE BALANCE DATABASE ────────────────────────────────────
# In production: HRMS leave balance API
LEAVE_BALANCE_DB = {
    "EMP-1042": {"CL": 5, "PL": 12, "SL": 8},
    "EMP-2087": {"CL": 3, "PL": 8,  "SL": 6},
    "EMP-0010": {"CL": 10, "PL": 15, "SL": 10},
    "EMP-9999": {"CL": 0,  "PL": 0,  "SL": 2},  # No balance (edge case)
}

# ── LEAVE HISTORY DATABASE ─────────────────────────────────────
# Last 3 months of leave records per employee
# In production: SQL query on leave_records table
today = datetime.now()
LEAVE_HISTORY_DB = {
    "EMP-1042": [
        # Normal: leaves on random days — should be LOW risk
        {"date": (today - timedelta(days=45)).strftime("%Y-%m-%d"), "day": "Wednesday", "type": "CL"},
        {"date": (today - timedelta(days=30)).strftime("%Y-%m-%d"), "day": "Tuesday",   "type": "PL"},
        {"date": (today - timedelta(days=15)).strftime("%Y-%m-%d"), "day": "Thursday",  "type": "SL"},
    ],
    "EMP-9999": [
        # Suspicious: took Monday leaves 3 times in 3 months — HIGH risk
        {"date": (today - timedelta(days=70)).strftime("%Y-%m-%d"), "day": "Monday", "type": "CL"},
        {"date": (today - timedelta(days=49)).strftime("%Y-%m-%d"), "day": "Monday", "type": "CL"},
        {"date": (today - timedelta(days=28)).strftime("%Y-%m-%d"), "day": "Monday", "type": "CL"},
        {"date": (today - timedelta(days=7)).strftime("%Y-%m-%d"),  "day": "Monday", "type": "CL"},
    ]
}

# ── POLICY STORE ───────────────────────────────────────────────
# In production: retrieved from Oracle Cloud vector store via RAG
# Here we simulate the retrieved chunk for each leave type
POLICY_STORE = {
    "CL": """
    CASUAL LEAVE (CL) POLICY — Company HR Policy v3.2
    
    Entitlement: 12 CL days per calendar year. Non-encashable. Lapses if unused by Dec 31.
    Maximum consecutive days: 3 days at a time.
    Notice period required: Minimum 1 working day prior notice (except emergencies).
    Approval: Direct manager must approve within 48 hours.
    Probation employees: Not eligible for CL during first 6 months.
    Restrictions: Cannot be combined with PL in the same application.
    Half-day option: Available (counts as 0.5 CL).
    """,
    "PL": """
    PRIVILEGE LEAVE (PL) POLICY — Company HR Policy v3.2
    
    Entitlement: 15 PL days per calendar year. Encashable up to 30 days after 3 years.
    Maximum consecutive days: 15 days at a time.
    Notice period required: Minimum 7 working days prior notice.
    Carry-forward: Up to 30 days can be carried forward to next year.
    Approval: Manager + HR must approve for PL > 5 days.
    Probation employees: Not eligible for PL during first 12 months.
    Restrictions: Requires handover/KT for any PL above 3 days.
    """,
    "SL": """
    SICK LEAVE (SL) POLICY — Company HR Policy v3.2
    
    Entitlement: 12 SL days per calendar year. Non-encashable. Non-transferable.
    Maximum consecutive days: 3 days without medical certificate. 5+ days requires certificate.
    Notice period: Not required for emergencies. Notify manager same day.
    Medical certificate: Mandatory if SL exceeds 3 consecutive days.
    Probation employees: Eligible for SL from day 1 (emergencies only).
    Restrictions: Cannot be taken immediately before or after a long weekend more than twice a year.
    """
}

# ── LEAVE RECORDS (approved leaves stored here) ────────────────
# In production: actual database table
APPROVED_LEAVES_DB = []  # Will be populated by Transaction Executor

# ── DATABASE HELPER FUNCTIONS ──────────────────────────────────

def get_employee(emp_id: str) -> Optional[Dict]:
    """Lookup employee by ID. Returns None if not found."""
    return EMPLOYEE_DB.get(emp_id)

def get_leave_balance(emp_id: str) -> Optional[Dict]:
    """Get leave balance for an employee."""
    return LEAVE_BALANCE_DB.get(emp_id)

def get_leave_history(emp_id: str, months: int = 3) -> List[Dict]:
    """Get leave history for last N months."""
    return LEAVE_HISTORY_DB.get(emp_id, [])

def get_policy(leave_type: str) -> str:
    """Retrieve policy text for a leave type."""
    return POLICY_STORE.get(leave_type, "Policy not found for this leave type.")

def save_leave_record(record: Dict) -> str:
    """Save approved leave to database. Returns record ID."""
    record_id = f"LV-{datetime.now().strftime('%Y%m%d%H%M%S')}-{len(APPROVED_LEAVES_DB)+1:04d}"
    record["record_id"] = record_id
    record["created_at"] = datetime.now().isoformat()
    APPROVED_LEAVES_DB.append(record)
    return record_id

def deduct_balance(emp_id: str, plan: Dict) -> bool:
    """Deduct approved leave from balance. Returns True if successful."""
    if emp_id not in LEAVE_BALANCE_DB:
        return False
    for leave_item in plan.get("breakdown", []):
        leave_type = leave_item["type"]
        days = leave_item["days"]
        if LEAVE_BALANCE_DB[emp_id].get(leave_type, 0) >= days:
            LEAVE_BALANCE_DB[emp_id][leave_type] -= days
        else:
            return False
    return True


print("✅ Mock databases initialized")
print(f"   Employees loaded: {len(EMPLOYEE_DB)}")
print(f"   Policies loaded: {list(POLICY_STORE.keys())}")
print()
print("📌 In production, replace these helper functions with real API calls")
print("📌 The node code stays IDENTICAL — only data sources change")

---
## 🛠️ CELL 4 — Utility Functions

### Theory: Separation of Concerns

Good production code separates **business logic** from **node logic**. Utility functions handle:
- Date calculations (working days, weekday names)
- Pattern detection algorithms
- Output formatters

This keeps each node clean and focused on its one job.

In [ ]:
# ============================================================
# CELL 4: Utility Functions
# ============================================================

def calculate_working_days(start_date_str: str, end_date_str: str) -> int:
    """
    Calculate working days between two dates (excluding weekends).
    In production, also exclude public holidays from a holiday calendar.
    """
    start = datetime.strptime(start_date_str, "%Y-%m-%d")
    end = datetime.strptime(end_date_str, "%Y-%m-%d")
    working_days = 0
    current = start
    while current <= end:
        if current.weekday() < 5:  # Monday=0, Friday=4
            working_days += 1
        current += timedelta(days=1)
    return working_days


def get_weekday_name(date_str: str) -> str:
    """Get the weekday name for a date string."""
    return datetime.strptime(date_str, "%Y-%m-%d").strftime("%A")


def detect_patterns(history: List[Dict], start_date: str) -> Dict:
    """
    Analyze leave history for suspicious patterns.
    
    Patterns detected:
    1. monday_pattern: Took Monday leave 3+ times in 3 months
    2. friday_pattern: Took Friday leave 3+ times in 3 months
    3. pre_holiday: Applying for leave just before a weekend/holiday
    4. high_frequency: More than 4 leaves in 3 months
    
    Returns dict with flags list and risk level.
    """
    flags = []
    
    if not history:
        return {"flags": [], "risk": "low", "summary": "No prior leave history found."}
    
    # Count by weekday
    weekday_counts = {"Monday": 0, "Tuesday": 0, "Wednesday": 0, 
                      "Thursday": 0, "Friday": 0}
    for record in history:
        day = record.get("day", "")
        if day in weekday_counts:
            weekday_counts[day] += 1
    
    # Check Monday pattern (3+ Mondays in 3 months is suspicious)
    if weekday_counts["Monday"] >= 3:
        flags.append("monday_pattern")
    
    # Check Friday pattern
    if weekday_counts["Friday"] >= 3:
        flags.append("friday_pattern")
    
    # Check high frequency (4+ leaves in 3 months)
    if len(history) >= 4:
        flags.append("high_frequency")
    
    # Check if current leave is on a Monday or Friday
    if start_date:
        weekday = get_weekday_name(start_date)
        if weekday in ["Monday", "Friday"] and weekday.lower()+"_pattern" in flags:
            flags.append(f"current_leave_is_{weekday.lower()}")
    
    # Determine risk level
    if len(flags) >= 2:
        risk = "high"
    elif len(flags) == 1:
        risk = "medium"
    else:
        risk = "low"
    
    # Build human-readable summary
    total_leaves = len(history)
    dominant_day = max(weekday_counts, key=weekday_counts.get)
    summary = (
        f"{total_leaves} leaves in last 3 months. "
        f"Most common day: {dominant_day} ({weekday_counts[dominant_day]} times). "
        f"Flags: {flags if flags else 'None'}."
    )
    
    return {"flags": flags, "risk": risk, "summary": summary}


def validate_kt_content(content: str) -> Dict:
    """Validate KT content meets minimum requirements."""
    if not content or not content.strip():
        return {"valid": False, "reason": "KT content cannot be empty", "word_count": 0}
    
    words = len(content.strip().split())
    if words < 50:
        return {
            "valid": False,
            "reason": f"KT content must be at least 50 words. Current: {words} words.",
            "word_count": words
        }
    
    return {"valid": True, "reason": "KT content meets requirements", "word_count": words}


def format_plan_for_display(plan: Dict) -> str:
    """Format a leave plan dict into readable text."""
    lines = [f"  📋 {plan.get('title', 'Plan')}"]
    for item in plan.get("breakdown", []):
        lines.append(f"     • {item['days']} days {item['type']}")
    lines.append(f"     Total: {plan.get('total_days', '?')} days")
    lines.append(f"     Why: {plan.get('benefit', '')}")
    return "\n".join(lines)


def print_node_header(node_name: str, node_num: str):
    """Consistent header for each node's output."""
    print(f"\n{'🔷'*3} NODE {node_num}: {node_name} {'🔷'*3}")
    print(f"⏱️  {datetime.now().strftime('%H:%M:%S')}")


def print_node_output(output: dict):
    """Print what a node is returning to state."""
    print(f"\n  📤 UPDATING STATE WITH:")
    for k, v in output.items():
        if k == "messages":
            print(f"     {k}: [+{len(v)} message(s)]")
        else:
            val_str = str(v)
            if len(val_str) > 100:
                val_str = val_str[:100] + "..."
            print(f"     {k}: {val_str}")


print("✅ Utility functions ready")
print()
# Quick test
test_days = calculate_working_days("2025-01-06", "2025-01-10")  # Mon to Fri
print(f"   Test — Working days Mon-Fri: {test_days} (expected: 5) ✅")
test_weekday = get_weekday_name("2025-01-06")
print(f"   Test — Weekday for 2025-01-06: {test_weekday} (expected: Monday) ✅")

---
## 🗣️ CELL 5 — Node 1: Intent Parser

### Theory: Structured Output Extraction with LLMs

**The Intent Parser** is the system's front door. It converts messy natural language ("I wanna take 3 days off from monday, CL") into clean structured data.

### Why prompt the LLM to return JSON?
We need the data in a predictable format so other nodes can use it reliably. The technique:
1. Instruct the LLM in the system prompt to return ONLY valid JSON
2. Define exactly what fields you need
3. Parse the JSON with `json.loads()`
4. Validate the result

### The Clarification Loop:
If `missing_info` is not empty after this node, the router sends the `clarification_question` back to the user and loops back to this node. This is how LangGraph enables multi-turn conversations within a single application.

In [ ]:
# ============================================================
# CELL 5: Node 1 — Intent Parser
# ============================================================

def intent_parser_node(state: LeaveApplicationState) -> dict:
    """
    NODE 1: INTENT PARSER
    
    Job: Extract structured leave data from the user's natural language message.
    
    Input  (reads from state): raw_input, messages
    Output (writes to state) : emp_id, emp_name, emp_email, manager_id, manager_email,
                               manager_name, leave_type, start_date, end_date, num_days,
                               reason, missing_info, clarification_question
    
    Routing after this node:
    - If missing_info is not empty → router asks user → comes back here
    - If all fields present → router sends to Pattern Analyser (N2)
    """
    print_node_header("INTENT PARSER", "1")
    print(f"  📥 Raw input received: '{state.get('raw_input', '')}'")
    
    # ── SYSTEM PROMPT ────────────────────────────────────────
    # This is the instruction set that controls LLM behavior.
    # Notice: explicit JSON format, all possible fields, handling of missing data.
    system_prompt = """
    You are an HR assistant that extracts leave application details from employee messages.
    
    Extract the following information and return ONLY a valid JSON object with these exact keys:
    {
        "emp_id": "employee ID if mentioned, else 'EMP-1042' as default for demo",
        "leave_type": "CL or PL or SL — infer from context if not stated",
        "start_date": "YYYY-MM-DD format, calculate from relative terms like 'monday', 'tomorrow', 'next week'",
        "end_date": "YYYY-MM-DD format",
        "num_days": "integer number of days",
        "reason": "stated reason or empty string",
        "missing_info": ["list of fields that are STILL missing after extraction"],
        "clarification_question": "a single friendly question to ask if missing_info is not empty"
    }
    
    Rules:
    - Today's date is: """ + datetime.now().strftime("%Y-%m-%d (%A)") + """
    - 'Monday' means the nearest upcoming Monday
    - Missing info should only include: start_date, end_date, num_days, leave_type
    - If num_days is given but no end_date, calculate end_date from start_date + num_days (working days)
    - If start_date and end_date given, don't list them as missing
    - Return ONLY the JSON object. No markdown. No explanation.
    """
    
    # ── LLM CALL ─────────────────────────────────────────────
    # We pass the conversation history so the LLM has context from previous turns
    messages_for_llm = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=state["raw_input"])
    ]
    
    print(f"  🤖 Calling LLM for intent extraction...")
    response = llm.invoke(messages_for_llm)
    raw_json = response.content.strip()
    
    # Remove markdown code blocks if LLM wraps in ```json
    if raw_json.startswith("```"):
        raw_json = re.sub(r'```json\n?|```', '', raw_json).strip()
    
    print(f"  🤖 LLM raw response: {raw_json[:200]}")
    
    # ── PARSE RESPONSE ────────────────────────────────────────
    try:
        parsed = json.loads(raw_json)
    except json.JSONDecodeError as e:
        print(f"  ❌ JSON parse error: {e}")
        # Return minimal error state
        return {
            "errors": [f"Intent Parser JSON error: {str(e)}"],
            "status": "error",
            "current_node": "intent_parser"
        }
    
    # ── ENRICH WITH EMPLOYEE DATA ──────────────────────────────
    # Look up employee details from our database
    emp_id = parsed.get("emp_id", "EMP-1042")
    employee = get_employee(emp_id)
    
    if not employee:
        return {
            "errors": [f"Employee {emp_id} not found in database"],
            "status": "error",
            "current_node": "intent_parser"
        }
    
    # Look up manager details
    manager = get_employee(employee["manager_id"])
    manager_name = manager["name"] if manager else "Unknown Manager"
    manager_email = manager["email"] if manager else "manager@company.com"
    
    # ── CALCULATE NUM_DAYS IF NEEDED ───────────────────────────
    num_days = parsed.get("num_days", 0)
    start_date = parsed.get("start_date", "")
    end_date = parsed.get("end_date", "")
    
    if start_date and end_date and not num_days:
        num_days = calculate_working_days(start_date, end_date)
        print(f"  📅 Calculated num_days from dates: {num_days}")
    
    # ── BUILD OUTPUT ───────────────────────────────────────────
    output = {
        "current_node": "intent_parser",
        "emp_id": emp_id,
        "emp_name": employee["name"],
        "emp_email": employee["email"],
        "manager_id": employee["manager_id"],
        "manager_email": manager_email,
        "manager_name": manager_name,
        "leave_type": parsed.get("leave_type", ""),
        "start_date": start_date,
        "end_date": end_date,
        "num_days": num_days,
        "reason": parsed.get("reason", ""),
        "missing_info": parsed.get("missing_info", []),
        "clarification_question": parsed.get("clarification_question", ""),
        "messages": [AIMessage(content=f"Intent parsed: {parsed.get('leave_type')} leave, {num_days} days, from {start_date}")]
    }
    
    # ── PRINT RESULTS ─────────────────────────────────────────
    print(f"\n  ✅ PARSED SUCCESSFULLY:")
    print(f"     Employee  : {output['emp_name']} ({output['emp_id']})")
    print(f"     Leave Type: {output['leave_type']}")
    print(f"     From      : {output['start_date']} ({get_weekday_name(start_date) if start_date else 'N/A'})")
    print(f"     To        : {output['end_date']}")
    print(f"     Days      : {output['num_days']}")
    print(f"     Manager   : {manager_name} ({manager_email})")
    print(f"     Missing   : {output['missing_info'] if output['missing_info'] else 'Nothing — all fields present!'}")
    
    if output['missing_info']:
        print(f"     ⚠️  Will ask: '{output['clarification_question']}'")
        print(f"     ⚠️  Router will loop back to this node after user responds")
    else:
        print(f"     ✅ All required fields present — routing to Pattern Analyser")
    
    print_node_output(output)
    return output


print("✅ Node 1 (Intent Parser) defined")
print("   Key design: LLM extracts JSON → we enrich with DB data → route based on missing_info")

---
## 📊 CELL 6 — Node 2: Pattern Analyser

### Theory: Rule-Based Analysis vs. LLM Analysis

Notice that Pattern Analyser uses **pure Python logic** (not an LLM call). This is intentional and important:

- **Use LLMs** for: understanding natural language, generating text, reasoning about ambiguous situations
- **Use Python/SQL** for: counting, comparing, filtering, deterministic rule-based checks

Pattern detection is deterministic ("Monday 3+ times = flag"), so an LLM would be:
- Overkill (adds cost and latency)
- Potentially inconsistent (LLMs can vary in threshold interpretation)
- Harder to audit ("why did the AI flag this?")

**Golden Rule**: Use the right tool for each job. Not everything in an AI system should use AI.

In [ ]:
# ============================================================
# CELL 6: Node 2 — Pattern Analyser
# ============================================================

def pattern_analyser_node(state: LeaveApplicationState) -> dict:
    """
    NODE 2: PATTERN ANALYSER
    
    Job: Detect suspicious leave patterns from historical data.
    Does NOT block the application — only flags for awareness.
    
    Input  (reads from state): emp_id, start_date
    Output (writes to state) : pattern_flags, pattern_risk, history_summary
    
    Routing after this node: ALWAYS → RAG Policy Reader (N3)
    (patterns don't block — they just inform later nodes)
    """
    print_node_header("PATTERN ANALYSER", "2")
    print(f"  📥 Analyzing patterns for: {state.get('emp_name')} ({state.get('emp_id')})")
    print(f"  📥 Requested date: {state.get('start_date')}")
    
    emp_id = state.get("emp_id", "")
    start_date = state.get("start_date", "")
    
    # ── FETCH HISTORY ────────────────────────────────────────
    # Currently: last 3 months. Future: last 12 months (just change the query)
    history = get_leave_history(emp_id, months=3)
    print(f"\n  📊 Leave history retrieved: {len(history)} records in last 3 months")
    
    if history:
        print("     History:")
        for record in history:
            print(f"       • {record['date']} ({record['day']}) — {record['type']}")
    else:
        print("     No leave history found (new employee or first leave)")
    
    # ── RUN PATTERN DETECTION ─────────────────────────────────
    # Pure Python logic — no LLM needed here
    result = detect_patterns(history, start_date)
    
    flags = result["flags"]
    risk = result["risk"]
    summary = result["summary"]
    
    # ── PRINT ANALYSIS ────────────────────────────────────────
    print(f"\n  🔍 PATTERN ANALYSIS RESULTS:")
    print(f"     Flags detected : {flags if flags else 'None ✅'}")
    print(f"     Risk level     : {risk.upper()} {'⚠️' if risk != 'low' else '✅'}")
    print(f"     Summary        : {summary}")
    
    if flags:
        print(f"\n  ⚠️  PATTERN DETECTED — This will be flagged in manager notification")
        print(f"     Note: Application is NOT blocked. Manager sees the flag.")
    else:
        print(f"\n  ✅ No suspicious patterns — proceeding normally")
    
    print(f"  ➡️  Routing: ALWAYS → RAG Policy Reader (N3)")
    
    output = {
        "current_node": "pattern_analyser",
        "pattern_flags": flags,
        "pattern_risk": risk,
        "history_summary": summary,
    }
    print_node_output(output)
    return output


print("✅ Node 2 (Pattern Analyser) defined")
print("   Key design: Pure Python logic — no LLM overhead for deterministic checks")

---
## 📚 CELL 7 — Node 3: RAG Policy Reader

### Theory: What is RAG (Retrieval-Augmented Generation)?

**RAG** = Retrieve relevant documents → Feed them to the LLM → Get a grounded answer

Why not just put the entire policy in the prompt?
- Policy documents can be hundreds of pages
- Context windows have limits
- It's expensive to send everything every time
- RAG retrieves only the **relevant** sections

### In Production:
```
1. Policy PDFs → chunked into paragraphs → embedded as vectors → stored in Oracle DB 23ai
2. At runtime: search query → vector similarity search → top 3 most relevant chunks
3. Chunks fed to LLM with the question → structured answer extracted
```

### In This Notebook:
We simulate step 2 (we already know which policy text to use based on leave_type). The LLM interaction in step 3 is real — the LLM genuinely reads the policy text and extracts rules.

In [ ]:
# ============================================================
# CELL 7: Node 3 — RAG Policy Reader
# ============================================================

def rag_policy_reader_node(state: LeaveApplicationState) -> dict:
    """
    NODE 3: RAG POLICY READER
    
    Job: Retrieve and interpret the relevant leave policy.
    Extracts structured rules that the Eligibility Checker and Optimizer will use.
    
    Input  (reads from state): leave_type, num_days, emp_id
    Output (writes to state) : policy_text, policy_summary, policy_rules
    
    Routing after this node: ALWAYS → Eligibility Checker (N4)
    """
    print_node_header("RAG POLICY READER", "3")
    leave_type = state.get("leave_type", "")
    num_days = state.get("num_days", 0)
    emp_id = state.get("emp_id", "")
    
    print(f"  📥 Querying policy for: {leave_type} leave, {num_days} days, employee {emp_id}")
    
    # ── STEP 1: RETRIEVE (simulated RAG retrieval) ────────────
    # In production: vector similarity search on Oracle DB 23ai / Pinecone
    # We simulate by selecting the correct policy section
    policy_text = get_policy(leave_type)
    print(f"\n  📄 RETRIEVED POLICY TEXT:")
    print(f"     {policy_text.strip()}")
    
    # ── STEP 2: GENERATE (LLM reads policy and extracts rules) ─
    # This is the 'G' in RAG — we feed retrieved text to LLM
    system_prompt = """
    You are an HR policy analyst. Read the provided leave policy text and extract the rules.
    
    Return ONLY a valid JSON object with these exact keys:
    {
        "summary": "2-3 sentence plain English summary of key rules",
        "max_consecutive_days": <integer>,
        "notice_days_required": <integer, 0 if same-day is allowed>,
        "requires_medical_cert": <true/false>,
        "encashable": <true/false>,
        "carry_forward_allowed": <true/false>,
        "probation_restricted": <true/false>,
        "kt_required_above_days": <integer, the threshold above which KT is needed, or 999 if never>,
        "special_restrictions": ["list of any special rules"]
    }
    
    Return ONLY the JSON. No markdown. No explanation.
    """
    
    user_prompt = f"""
    Leave Type: {leave_type}
    Employee requesting {num_days} days.
    
    Policy text:
    {policy_text}
    
    Extract and structure the rules.
    """
    
    print(f"\n  🤖 Calling LLM to extract structured rules from policy...")
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])
    
    raw_json = response.content.strip()
    if raw_json.startswith("```"):
        raw_json = re.sub(r'```json\n?|```', '', raw_json).strip()
    
    try:
        policy_rules = json.loads(raw_json)
    except json.JSONDecodeError:
        # Fallback defaults if LLM output can't be parsed
        policy_rules = {
            "summary": f"Standard {leave_type} policy applies.",
            "max_consecutive_days": 3,
            "notice_days_required": 1,
            "requires_medical_cert": False,
            "probation_restricted": True,
            "kt_required_above_days": 3,
            "special_restrictions": []
        }
    
    # ── PRINT RESULTS ─────────────────────────────────────────
    print(f"\n  ✅ STRUCTURED POLICY RULES EXTRACTED:")
    print(f"     Summary          : {policy_rules.get('summary', 'N/A')}")
    print(f"     Max consecutive  : {policy_rules.get('max_consecutive_days', '?')} days")
    print(f"     Notice required  : {policy_rules.get('notice_days_required', '?')} working day(s)")
    print(f"     Probation blocked: {policy_rules.get('probation_restricted', '?')}")
    print(f"     KT needed after  : {policy_rules.get('kt_required_above_days', '?')} days")
    print(f"     Special rules    : {policy_rules.get('special_restrictions', [])}")
    print(f"  ➡️  Routing: ALWAYS → Eligibility Checker (N4)")
    
    output = {
        "current_node": "rag_policy_reader",
        "policy_text": policy_text,
        "policy_summary": policy_rules.get("summary", ""),
        "policy_rules": policy_rules,
    }
    print_node_output(output)
    return output


print("✅ Node 3 (RAG Policy Reader) defined")
print("   Key design: Retrieve text first, then LLM extracts structured rules from it")

---
## ✅ CELL 8 — Node 4: Eligibility Checker

### Theory: The First Critical Conditional Edge

After this node, the graph **branches** for the first time:
- `is_eligible = True` → continue to Optimizer
- `is_eligible = False` → END the graph with an error message

This is a **conditional edge** in LangGraph terms. The router function reads `state['is_eligible']` and returns the name of the next node accordingly.

### Multiple Check Rules:
The node checks several conditions in sequence — if any fails, the application stops. This is the "fail fast" principle: don't waste the user's time showing plan options if they're not eligible.

In [ ]:
# ============================================================
# CELL 8: Node 4 — Eligibility Checker
# ============================================================

def eligibility_checker_node(state: LeaveApplicationState) -> dict:
    """
    NODE 4: ELIGIBILITY CHECKER
    
    Job: Multi-rule eligibility validation.
    This is a CRITICAL node — if it fails, the application ends here.
    
    Input  (reads from state): emp_id, leave_type, num_days, start_date, policy_rules
    Output (writes to state) : balance_CL, balance_PL, balance_SL, is_eligible, ineligibility_reason
    
    Routing after this node:
    - is_eligible = True  → Leave Optimizer (N5)
    - is_eligible = False → END (show error to employee)
    """
    print_node_header("ELIGIBILITY CHECKER", "4")
    emp_id = state.get("emp_id", "")
    leave_type = state.get("leave_type", "")
    num_days = state.get("num_days", 0)
    policy_rules = state.get("policy_rules", {})
    
    print(f"  📥 Checking eligibility for {state.get('emp_name')} ({emp_id})")
    print(f"  📥 Requesting: {num_days} days of {leave_type}")
    
    # ── FETCH BALANCE ─────────────────────────────────────────
    balance = get_leave_balance(emp_id)
    employee = get_employee(emp_id)
    
    if not balance:
        output = {
            "current_node": "eligibility_checker",
            "balance_CL": 0, "balance_PL": 0, "balance_SL": 0,
            "is_eligible": False,
            "ineligibility_reason": f"Could not fetch leave balance for {emp_id}"
        }
        print(f"  ❌ Cannot fetch balance for {emp_id}")
        print_node_output(output)
        return output
    
    print(f"\n  💰 CURRENT LEAVE BALANCES:")
    print(f"     CL Balance: {balance['CL']} days")
    print(f"     PL Balance: {balance['PL']} days")
    print(f"     SL Balance: {balance['SL']} days")
    
    # ── ELIGIBILITY CHECKS (in order, fail fast) ──────────────
    print(f"\n  🔍 RUNNING ELIGIBILITY CHECKS:")
    
    is_eligible = True
    reason = ""
    
    # CHECK 1: Probation restriction
    if employee and employee.get("probation") and policy_rules.get("probation_restricted", True):
        is_eligible = False
        reason = f"{leave_type} leave is not available during probation period."
        print(f"     ❌ CHECK 1 (Probation): FAILED — {reason}")
    else:
        print(f"     ✅ CHECK 1 (Probation): PASSED — Not on probation")
    
    # CHECK 2: Sufficient balance
    if is_eligible:
        available = balance.get(leave_type, 0)
        if available < num_days:
            is_eligible = False
            reason = f"Insufficient {leave_type} balance. Requested: {num_days} days, Available: {available} days."
            print(f"     ❌ CHECK 2 (Balance): FAILED — {reason}")
        else:
            print(f"     ✅ CHECK 2 (Balance): PASSED — {available} available, {num_days} requested")
    
    # CHECK 3: Max consecutive days policy
    if is_eligible:
        max_allowed = policy_rules.get("max_consecutive_days", 999)
        if num_days > max_allowed:
            is_eligible = False
            reason = f"Maximum {leave_type} allowed at once is {max_allowed} days. You requested {num_days}."
            print(f"     ❌ CHECK 3 (Max days): FAILED — {reason}")
        else:
            print(f"     ✅ CHECK 3 (Max days): PASSED — {num_days} ≤ max {max_allowed} days")
    
    # CHECK 4: Notice period
    if is_eligible:
        start_date = state.get("start_date", "")
        notice_required = policy_rules.get("notice_days_required", 0)
        if start_date and notice_required > 0:
            days_until_leave = (datetime.strptime(start_date, "%Y-%m-%d") - datetime.now()).days
            if days_until_leave < notice_required:
                # For SL (emergency), we allow same-day but add a warning
                if leave_type != "SL":
                    is_eligible = False
                    reason = f"Notice period not met. {notice_required} working day(s) required. Your leave starts in {max(0,days_until_leave)} day(s)."
                    print(f"     ❌ CHECK 4 (Notice): FAILED — {reason}")
                else:
                    print(f"     ⚠️  CHECK 4 (Notice): WAIVED — SL can be applied same-day")
            else:
                print(f"     ✅ CHECK 4 (Notice): PASSED — {days_until_leave} days notice given")
        else:
            print(f"     ✅ CHECK 4 (Notice): SKIPPED — No notice requirement")
    
    # ── FINAL VERDICT ─────────────────────────────────────────
    print(f"\n  {'✅ ELIGIBLE' if is_eligible else '❌ INELIGIBLE'}: {reason if not is_eligible else 'All checks passed!'}")
    print(f"  ➡️  Routing: {'→ Leave Optimizer (N5)' if is_eligible else '→ END (show error to employee)'}")
    
    output = {
        "current_node": "eligibility_checker",
        "balance_CL": balance.get("CL", 0),
        "balance_PL": balance.get("PL", 0),
        "balance_SL": balance.get("SL", 0),
        "is_eligible": is_eligible,
        "ineligibility_reason": reason,
        "status": "in_progress" if is_eligible else "rejected",
        "messages": [AIMessage(content=f"Eligibility: {'Approved to proceed' if is_eligible else reason}")] 
    }
    print_node_output(output)
    return output


print("✅ Node 4 (Eligibility Checker) defined")
print("   Key design: Multiple ordered checks, fail-fast — first failure ends eligibility")

---
## 💡 CELL 9 — Node 5: Leave Plan Optimizer (CRITICAL)

### Theory: This is the Heart of the System

The optimizer is where the AI adds real value. Instead of just approving or rejecting, it **actively helps the employee use their leave wisely**.

**How it generates multiple plans:**
1. It knows the employee's balances (from N4)
2. It knows the policy constraints (from N3)
3. It knows any pattern flags (from N2)
4. The LLM generates 3-5 creative combinations that stay within constraints

**In the notebook**, we simulate the user selecting a plan (in production, this would be a UI event). The simulated selection updates `selected_plan` and `plan_selected = True`.

### Second HITL Point:
In production, after displaying cards, the graph PAUSES (via Redis checkpoint) waiting for the employee to click. For this notebook, we auto-select plan 0 with a comment showing where real UI interaction happens.

In [ ]:
# ============================================================
# CELL 9: Node 5 — Leave Plan Optimizer (CRITICAL)
# ============================================================

def leave_optimizer_node(state: LeaveApplicationState) -> dict:
    """
    NODE 5: LEAVE PLAN OPTIMIZER — The heart of the system
    
    Job: Generate 3-5 intelligent leave plan options for the employee to choose from.
    Uses LLM to reason about leave type combinations, balances, and policy constraints.
    
    Input  (reads from state): leave_type, num_days, start_date, end_date,
                               balance_CL/PL/SL, policy_rules, pattern_flags
    Output (writes to state) : leave_plan_options, selected_plan, plan_selected
    
    Routing after this node:
    - plan_selected=True AND num_days >= 3 → KT Validator (N6)
    - plan_selected=True AND num_days <  3 → Notification Agent (N7)
    
    NOTE ON HITL:
    In production, graph PAUSES after displaying cards and waits for user click.
    In this notebook, we auto-select the first plan for demonstration.
    See comment block below for production implementation.
    """
    print_node_header("LEAVE PLAN OPTIMIZER", "5")
    
    leave_type  = state.get("leave_type", "")
    num_days    = state.get("num_days", 0)
    start_date  = state.get("start_date", "")
    end_date    = state.get("end_date", "")
    bal_CL      = state.get("balance_CL", 0)
    bal_PL      = state.get("balance_PL", 0)
    bal_SL      = state.get("balance_SL", 0)
    policy_rules = state.get("policy_rules", {})
    pattern_flags = state.get("pattern_flags", [])
    
    print(f"  📥 Generating plans for {num_days} days of {leave_type}")
    print(f"  📥 Available: {bal_CL} CL | {bal_PL} PL | {bal_SL} SL")
    print(f"  📥 Pattern flags: {pattern_flags if pattern_flags else 'None'}")
    
    # ── LLM GENERATES PLANS ────────────────────────────────────
    system_prompt = """
    You are an intelligent leave management assistant. Generate optimal leave plan options.
    
    Return ONLY a valid JSON array of 3-4 plan objects. Each object must have:
    {
        "plan_id": "plan_A" (or B, C, D),
        "title": "short memorable name for this plan",
        "breakdown": [{"type": "CL", "days": 3}, {"type": "PL", "days": 2}],
        "total_days": <integer>,
        "benefit": "one sentence explaining WHY this plan is smart for the employee",
        "risk_note": "any caveat or thing to be aware of, empty string if none"
    }
    
    Rules:
    - Each plan must cover the requested number of days
    - Each plan must use ONLY available balance (never exceed it)
    - Plans must comply with max_consecutive_days policy
    - Vary the plans meaningfully (not just slightly different)
    - If there are pattern flags, avoid plans that make patterns worse
    - Make plans genuinely useful, not arbitrary
    
    Return ONLY the JSON array. No markdown. No explanation.
    """
    
    user_prompt = f"""
    Employee wants: {num_days} days of {leave_type}
    From: {start_date} To: {end_date}
    
    Leave balances:
    - CL available: {bal_CL} days
    - PL available: {bal_PL} days
    - SL available: {bal_SL} days
    
    Policy constraints:
    - Max consecutive {leave_type}: {policy_rules.get('max_consecutive_days', 99)} days
    - CL cannot be combined with PL in same application: {policy_rules.get('special_restrictions', [])}
    
    Pattern concerns: {pattern_flags if pattern_flags else 'None'}
    
    Generate 3-4 varied, intelligent leave plan options.
    """
    
    print(f"\n  🤖 Calling LLM to generate plan options...")
    response = llm_creative.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])
    
    raw_json = response.content.strip()
    if raw_json.startswith("```"):
        raw_json = re.sub(r'```json\n?|```', '', raw_json).strip()
    
    try:
        plans = json.loads(raw_json)
        if not isinstance(plans, list):
            plans = [plans]
    except json.JSONDecodeError:
        # Fallback: create a simple default plan
        plans = [{
            "plan_id": "plan_A",
            "title": "Standard Plan",
            "breakdown": [{"type": leave_type, "days": num_days}],
            "total_days": num_days,
            "benefit": f"Uses {num_days} {leave_type} days as requested.",
            "risk_note": ""
        }]
    
    # ── DISPLAY PLANS TO EMPLOYEE ──────────────────────────────
    print(f"\n  💡 GENERATED {len(plans)} LEAVE PLAN OPTIONS:")
    print(f"  {'─'*50}")
    for i, plan in enumerate(plans):
        print(f"\n  📋 OPTION {i+1}: {plan.get('title', 'Plan')} ({plan.get('plan_id')})")
        print(f"     Breakdown:")
        for item in plan.get("breakdown", []):
            print(f"       • {item['days']} day(s) of {item['type']}")
        print(f"     Total    : {plan.get('total_days', num_days)} days")
        print(f"     Why      : {plan.get('benefit', '')}")
        if plan.get('risk_note'):
            print(f"     ⚠️  Note  : {plan.get('risk_note')}")
    
    print(f"\n  {'─'*50}")
    
    # ── SIMULATE USER SELECTION ────────────────────────────────
    # ╔══════════════════════════════════════════════════════════╗
    # ║  PRODUCTION HITL IMPLEMENTATION:                        ║
    # ║                                                          ║
    # ║  1. Display cards in UI (React/Vue frontend)            ║
    # ║  2. User clicks a card → POST /api/select-plan          ║
    # ║  3. API calls graph.resume(thread_id, {                 ║
    # ║       "selected_plan": plans[clicked_index],            ║
    # ║       "plan_selected": True                             ║
    # ║     })                                                   ║
    # ║  4. Graph continues from checkpoint                     ║
    # ║                                                          ║
    # ║  For this notebook: auto-select plan 0 (first option)   ║
    # ╚══════════════════════════════════════════════════════════╝
    
    # Simulate employee selecting the first plan
    SIMULATED_USER_SELECTED_PLAN_INDEX = 0  # Change this to test different plans
    selected_plan = plans[SIMULATED_USER_SELECTED_PLAN_INDEX]
    
    print(f"  👤 SIMULATED USER ACTION: Employee selected Plan {SIMULATED_USER_SELECTED_PLAN_INDEX + 1}")
    print(f"     → Selected: '{selected_plan.get('title')}'")
    print(f"     → In production: graph pauses here, user clicks card in UI")
    
    kt_needed = num_days >= 3
    print(f"\n  ➡️  Routing: num_days={num_days} >= 3? {kt_needed}")
    print(f"     → {'KT Validator (N6)' if kt_needed else 'Notification Agent (N7) — KT not required'}")
    
    output = {
        "current_node": "leave_optimizer",
        "leave_plan_options": plans,
        "selected_plan": selected_plan,
        "plan_selected": True,
        "kt_required": kt_needed,
        "messages": [AIMessage(content=f"Plan selected: {selected_plan.get('title')} — {selected_plan.get('benefit')}")]
    }
    print_node_output(output)
    return output


print("✅ Node 5 (Leave Plan Optimizer) defined — the most important node!")
print("   Key design: LLM generates smart plan combinations → user picks one → graph routes based on days")

In [ ]:
# ============================================================
# CELL 10: Node 6 — KT Validator (HITL)
# ============================================================

def kt_validator_node(state: LeaveApplicationState) -> dict:
    """
    NODE 6: KT VALIDATOR (Human-in-the-Loop)
    
    Job: Collect and validate Knowledge Transfer details when leave >= 3 days.
    Only runs if num_days >= 3.
    
    Input  (reads from state): emp_id, emp_name, num_days
    Output (writes to state) : kt_required, kt_receiver_id, kt_receiver_name,
                               kt_content, kt_valid
    
    Routing after this node:
    - kt_valid = True  → Notification Agent (N7)
    - kt_valid = False → would loop back (in production, re-show form)
    """
    print_node_header("KT VALIDATOR (HITL)", "6")
    emp_id = state.get("emp_id", "")
    num_days = state.get("num_days", 0)
    
    print(f"  📥 KT required because num_days={num_days} >= 3")
    print(f"  📥 Employee {state.get('emp_name')} ({emp_id}) must complete KT form")
    
    # ── SIMULATED KT FORM SUBMISSION ──────────────────────────
    # ╔══════════════════════════════════════════════════════════╗
    # ║  PRODUCTION IMPLEMENTATION:                             ║
    # ║  1. Display KT form in UI                              ║
    # ║  2. Employee fills: receiver_id + KT content           ║
    # ║  3. Live word count validation on frontend             ║
    # ║  4. Live employee lookup as they type receiver ID      ║
    # ║  5. Submit → POST /api/submit-kt                       ║
    # ║  6. API validates → graph.resume(thread_id, kt_data)   ║
    # ╚══════════════════════════════════════════════════════════╝
    
    # Simulated input (what employee would type in the form)
    SIMULATED_KT = {
        "receiver_id": "EMP-2087",
        "content": (
            "I am handing over the Q4 analytics dashboard project which is currently in the "
            "testing phase. Amit should check the three pending pull requests on GitHub — "
            "PR #287 for the data pipeline fix, PR #291 for the UI refresh, and PR #296 "
            "for the export feature. The weekly client sync call is on Wednesday at 3 PM, "
            "the deck is in our shared Google Drive under Client Meetings folder dated this week. "
            "Database credentials are in the team Vault. Do not run the migration script until "
            "the DevOps team gives the green light — they know about it. Slack Amit if anything "
            "urgent comes up, or DM me on personal number which is in the team directory."
        )
    }
    
    print(f"\n  📝 SIMULATED KT FORM SUBMISSION:")
    print(f"     KT Receiver ID : {SIMULATED_KT['receiver_id']}")
    print(f"     KT Content     : {SIMULATED_KT['content'][:100]}...")
    
    # ── VALIDATE RECEIVER ─────────────────────────────────────
    print(f"\n  🔍 VALIDATING KT:")
    receiver = get_employee(SIMULATED_KT["receiver_id"])
    
    if not receiver:
        print(f"  ❌ Receiver {SIMULATED_KT['receiver_id']} not found in employee database")
        output = {
            "current_node": "kt_validator",
            "kt_required": True,
            "kt_valid": False,
            "kt_receiver_id": SIMULATED_KT["receiver_id"],
            "kt_receiver_name": "Not Found",
            "kt_content": SIMULATED_KT["content"]
        }
        print_node_output(output)
        return output
    
    print(f"     ✅ Receiver found: {receiver['name']} ({receiver['email']})")
    
    # ── VALIDATE CONTENT ──────────────────────────────────────
    validation = validate_kt_content(SIMULATED_KT["content"])
    word_count = validation["word_count"]
    
    if not validation["valid"]:
        print(f"  ❌ Word count check FAILED: {validation['reason']}")
        output = {
            "current_node": "kt_validator",
            "kt_required": True,
            "kt_valid": False,
            "kt_receiver_id": SIMULATED_KT["receiver_id"],
            "kt_receiver_name": receiver["name"],
            "kt_content": SIMULATED_KT["content"]
        }
        print_node_output(output)
        return output
    
    print(f"     ✅ Word count: {word_count} words (minimum 50 required)")
    print(f"     ✅ KT validation PASSED")
    print(f"  ➡️  Routing: → Notification Agent (N7)")
    
    output = {
        "current_node": "kt_validator",
        "kt_required": True,
        "kt_receiver_id": SIMULATED_KT["receiver_id"],
        "kt_receiver_name": receiver["name"],
        "kt_content": SIMULATED_KT["content"],
        "kt_valid": True,
        "messages": [AIMessage(content=f"KT validated: {word_count} words. Receiver: {receiver['name']}")]
    }
    print_node_output(output)
    return output


print("✅ Node 6 (KT Validator) defined")

In [ ]:
# ============================================================
# CELL 11: Node 7 — Notification Agent
# ============================================================

def notification_agent_node(state: LeaveApplicationState) -> dict:
    """
    NODE 7: NOTIFICATION AGENT
    
    Job: Compose a comprehensive email to the manager and simulate sending it.
    In production: PAUSES here using Redis thread, waiting for manager's approval webhook.
    In this notebook: simulates immediate approval to complete the demonstration.
    
    Input  (reads from state): All employee, leave, plan, and KT details
    Output (writes to state) : notification_sent, email_content, approval_status
    
    Routing after this node:
    - approval_status = 'approved' → Transaction Executor (N8)
    - approval_status = 'rejected' → END
    """
    print_node_header("NOTIFICATION AGENT", "7")
    
    # Gather all relevant data from state
    emp_name     = state.get("emp_name", "")
    emp_id       = state.get("emp_id", "")
    emp_email    = state.get("emp_email", "")
    manager_name = state.get("manager_name", "")
    manager_email= state.get("manager_email", "")
    leave_type   = state.get("leave_type", "")
    start_date   = state.get("start_date", "")
    end_date     = state.get("end_date", "")
    num_days     = state.get("num_days", 0)
    selected_plan= state.get("selected_plan", {})
    pattern_flags= state.get("pattern_flags", [])
    pattern_risk = state.get("pattern_risk", "low")
    kt_required  = state.get("kt_required", False)
    kt_receiver  = state.get("kt_receiver_name", "")
    kt_content   = state.get("kt_content", "")
    bal_CL       = state.get("balance_CL", 0)
    bal_PL       = state.get("balance_PL", 0)
    bal_SL       = state.get("balance_SL", 0)
    
    print(f"  📥 Composing notification for manager: {manager_name} ({manager_email})")
    
    # ── BUILD EMAIL CONTENT ────────────────────────────────────
    # Format the selected plan breakdown
    plan_breakdown = ""
    for item in selected_plan.get("breakdown", []):
        plan_breakdown += f"  - {item['days']} day(s) of {item['type']}\n"
    
    # Format pattern warning
    pattern_section = ""
    if pattern_flags:
        pattern_section = f"""
⚠️  PATTERN ALERT ({pattern_risk.upper()} RISK):
   Flags detected: {', '.join(pattern_flags)}
   This is for your awareness — not an automatic block.
"""
    else:
        pattern_section = "✅ Pattern Analysis: No suspicious patterns detected."
    
    # Format KT section
    kt_section = ""
    if kt_required:
        kt_section = f"""
KNOWLEDGE TRANSFER:
  KT Given to : {kt_receiver}
  KT Content  : {kt_content[:200]}...
"""
    else:
        kt_section = "KT: Not required (leave < 3 days)"
    
    # Calculate balance after approval
    balance_after = {"CL": bal_CL, "PL": bal_PL, "SL": bal_SL}
    for item in selected_plan.get("breakdown", []):
        lt = item["type"]
        balance_after[lt] = balance_after.get(lt, 0) - item["days"]
    
    email_content = f"""{'='*60}
FROM    : LeaveFlow AI System <leaveflow@company.com>
TO      : {manager_name} <{manager_email}>
SUBJECT : LEAVE REQUEST — {emp_name.upper()} · {num_days} DAYS · {leave_type}
{'='*60}

Dear {manager_name},

{emp_name} ({emp_id}) has submitted a leave request through the automated LeaveFlow AI system. 
Please review the details below and use the APPROVE or REJECT buttons at the bottom.

─── EMPLOYEE DETAILS ─────────────────────────────────────
  Employee   : {emp_name}
  ID         : {emp_id}
  Email      : {emp_email}

─── LEAVE DETAILS ────────────────────────────────────────
  Leave Type : {leave_type} ({selected_plan.get('title', 'Standard')})
  From       : {start_date} ({get_weekday_name(start_date) if start_date else ''})
  To         : {end_date}
  Total Days : {num_days} working days

  Plan Breakdown:
{plan_breakdown}
  Why this plan: {selected_plan.get('benefit', '')}

─── LEAVE BALANCE (if approved) ──────────────────────────
  CL: {bal_CL} → {balance_after['CL']} remaining
  PL: {bal_PL} → {balance_after['PL']} remaining  
  SL: {bal_SL} → {balance_after['SL']} remaining

─── PATTERN ANALYSIS ─────────────────────────────────────
{pattern_section}

─── KNOWLEDGE TRANSFER ───────────────────────────────────
{kt_section}

─────────────────────────────────────────────────────────
  [ ✅ APPROVE ]    [ ❌ REJECT ]
  https://company.com/leave/approve?token=DEMO_TOKEN_12345
  https://company.com/leave/reject?token=DEMO_TOKEN_12345
─────────────────────────────────────────────────────────

This email was generated by LeaveFlow AI. Please do not reply.
{'='*60}"""
    
    # ── DISPLAY THE EMAIL ─────────────────────────────────────
    print(f"\n  📧 EMAIL COMPOSED AND SENT:")
    print(email_content)
    
    # ── SIMULATE MANAGER APPROVAL ─────────────────────────────
    # ╔══════════════════════════════════════════════════════════╗
    # ║  PRODUCTION REDIS PAUSE IMPLEMENTATION:                 ║
    # ║                                                          ║
    # ║  After email.send():                                    ║
    # ║    thread_id = graph.get_state(config).thread_id        ║
    # ║    redis.set(f"leave:{token}", thread_id)               ║
    # ║    raise NodeInterrupt("Waiting for manager approval")  ║
    # ║                                                          ║
    # ║  When manager clicks Approve:                           ║
    # ║    POST /api/leave/approve?token=TOKEN                  ║
    # ║    thread_id = redis.get(f"leave:{token}")              ║
    # ║    graph.invoke(None, config={"thread_id": thread_id},  ║
    # ║      interrupt_update={"approval_status": "approved"})  ║
    # ╚══════════════════════════════════════════════════════════╝
    
    # For notebook: simulate manager approving
    SIMULATED_APPROVAL = "approved"  # Change to "rejected" to test rejection path
    
    print(f"\n  ⏸️  PRODUCTION: Graph would PAUSE here for manager action")
    print(f"  🔄 NOTEBOOK SIMULATION: Manager response = '{SIMULATED_APPROVAL}'")
    print(f"  ➡️  Routing: approval_status='{SIMULATED_APPROVAL}' → {'Transaction Executor (N8)' if SIMULATED_APPROVAL=='approved' else 'END (rejected)'}")
    
    output = {
        "current_node": "notification_agent",
        "email_content": email_content,
        "notification_sent": True,
        "approval_status": SIMULATED_APPROVAL,
        "messages": [AIMessage(content=f"Notification sent to {manager_name}. Status: {SIMULATED_APPROVAL}")]
    }
    print_node_output(output)
    return output


print("✅ Node 7 (Notification Agent) defined")
print("   Key design: Composes rich email, simulates HITL pause, routes based on approval")

In [ ]:
# ============================================================
# CELL 12: Node 8 — Transaction Executor
# ============================================================

def transaction_executor_node(state: LeaveApplicationState) -> dict:
    """
    NODE 8: TRANSACTION EXECUTOR
    
    Job: Commit everything to databases atomically. The final node.
    Only runs after manager approves.
    
    Input  (reads from state): Full state — all fields
    Output (writes to state) : final_record_id, confirmation_sent, status
    
    Routing after this node: END (success)
    """
    print_node_header("TRANSACTION EXECUTOR", "8")
    print(f"  📥 Manager approved — committing to all databases")
    
    emp_id       = state.get("emp_id", "")
    selected_plan= state.get("selected_plan", {})
    
    # ── BUILD COMPLETE LEAVE RECORD ────────────────────────────
    leave_record = {
        "emp_id":           state.get("emp_id"),
        "emp_name":         state.get("emp_name"),
        "emp_email":        state.get("emp_email"),
        "manager_id":       state.get("manager_id"),
        "leave_type":       state.get("leave_type"),
        "start_date":       state.get("start_date"),
        "end_date":         state.get("end_date"),
        "num_days":         state.get("num_days"),
        "plan_selected":    state.get("selected_plan"),
        "pattern_flags":    state.get("pattern_flags", []),
        "kt_required":      state.get("kt_required", False),
        "kt_receiver_name": state.get("kt_receiver_name", ""),
        "kt_content":       state.get("kt_content", ""),
        "approval_status":  "approved",
        "approved_by":      state.get("manager_id"),
    }
    
    print(f"\n  💾 EXECUTING DATABASE WRITES (in atomic transaction):")
    
    # WRITE 1: Save leave record
    record_id = save_leave_record(leave_record)
    print(f"     ✅ [1/3] Leave record saved — ID: {record_id}")
    
    # WRITE 2: Deduct balance
    deduct_success = deduct_balance(emp_id, selected_plan)
    new_balance = get_leave_balance(emp_id)
    print(f"     {'✅' if deduct_success else '❌'} [2/3] Balance deducted — New balance: CL={new_balance.get('CL')}, PL={new_balance.get('PL')}, SL={new_balance.get('SL')}")
    
    # WRITE 3: Simulate calendar block
    print(f"     ✅ [3/3] Calendar blocked: {state.get('start_date')} to {state.get('end_date')}")
    
    # SEND CONFIRMATION EMAIL TO EMPLOYEE
    confirmation = f"""
  ✉️  CONFIRMATION EMAIL TO EMPLOYEE:
  To: {state.get('emp_name')} <{state.get('emp_email')}>
  Subject: ✅ Your Leave is Approved — {record_id}
  
  Dear {state.get('emp_name')},
  
  Your leave request has been approved by {state.get('manager_name')}.
  
  Details:
    Record ID  : {record_id}
    Leave Type : {state.get('leave_type')}
    From       : {state.get('start_date')}
    To         : {state.get('end_date')}
    Days       : {state.get('num_days')}
    Plan       : {selected_plan.get('title')}
  
  Have a great break! 🌴
    """
    print(confirmation)
    
    print(f"  ✅ ALL DONE! Leave application complete.")
    print(f"  ➡️  Routing: → END")
    
    output = {
        "current_node": "transaction_executor",
        "final_record_id": record_id,
        "confirmation_sent": True,
        "status": "complete",
        "messages": [AIMessage(content=f"Leave approved and saved. Record: {record_id}. Your leave is confirmed! 🎉")]
    }
    print_node_output(output)
    return output


print("✅ Node 8 (Transaction Executor) defined — final node")

---
## 🔀 CELL 13 — Routing Functions (Conditional Edges)

### Theory: How Conditional Edges Work in LangGraph

In LangGraph, a **conditional edge** is a Python function that:
1. Takes the current state as input
2. Returns a **string** — the name of the next node (or `END`)

```python
graph.add_conditional_edges(
    "source_node",     # Which node this comes after
    my_router_function, # Function that decides where to go
    {                   # Map of return values to node names
        "go_to_A": "node_A",
        "go_to_B": "node_B",
        "end": END
    }
)
```

The router function reads the state and returns one of the keys in the mapping. LangGraph routes to the corresponding node.

### All Conditional Edges in This System:
1. **After Intent Parser** → loop (missing info) or continue (all info present)
2. **After Eligibility Checker** → optimizer (eligible) or end (ineligible)  
3. **After Leave Optimizer** → KT validator (≥3 days) or notification (< 3 days)
4. **After KT Validator** → notification (valid) or retry (invalid)
5. **After Notification Agent** → transaction (approved) or end (rejected)

In [ ]:
# ============================================================
# CELL 13: Routing Functions — All Conditional Edges
# ============================================================

def route_after_intent_parser(state: LeaveApplicationState) -> str:
    """
    ROUTER 1: After Intent Parser
    
    Decision: Did we get all the info we need?
    - Yes (missing_info is empty) → proceed to pattern analysis
    - No  (missing_info has items) → ask user, loop back
    - Error → end with error
    """
    if state.get("status") == "error":
        print(f"  🔀 ROUTER (intent_parser → error): status=error")
        return "end_with_error"
    
    missing = state.get("missing_info", [])
    if missing:
        # In production: send clarification_question to user, wait for response
        # In this notebook: this path is demonstrated in the edge case run
        print(f"  🔀 ROUTER (intent_parser → clarify): missing_info={missing}")
        return "need_clarification"
    
    print(f"  🔀 ROUTER (intent_parser → pattern_analyser): all fields present")
    return "proceed_to_pattern"


def route_after_eligibility(state: LeaveApplicationState) -> str:
    """
    ROUTER 2: After Eligibility Checker
    
    Decision: Is the employee eligible?
    - is_eligible = True  → continue to plan generation
    - is_eligible = False → stop, show reason to employee
    """
    is_eligible = state.get("is_eligible", False)
    if is_eligible:
        print(f"  🔀 ROUTER (eligibility → leave_optimizer): employee is eligible ✅")
        return "eligible"
    else:
        reason = state.get("ineligibility_reason", "Unknown reason")
        print(f"  🔀 ROUTER (eligibility → end): NOT eligible — {reason}")
        return "ineligible"


def route_after_optimizer(state: LeaveApplicationState) -> str:
    """
    ROUTER 3: After Leave Optimizer (after user selects a plan)
    
    Decision: Does the employee need to do KT?
    - num_days >= 3 → KT is required → go to KT validator
    - num_days <  3 → No KT needed  → go straight to notification
    """
    num_days = state.get("num_days", 0)
    plan_selected = state.get("plan_selected", False)
    
    if not plan_selected:
        # Guard: shouldn't happen, but handle gracefully
        print(f"  🔀 ROUTER (optimizer → wait): plan not yet selected")
        return "wait_for_selection"
    
    if num_days >= 3:
        print(f"  🔀 ROUTER (optimizer → kt_validator): {num_days} days ≥ 3, KT required")
        return "kt_required"
    else:
        print(f"  🔀 ROUTER (optimizer → notification): {num_days} days < 3, skip KT")
        return "no_kt_needed"


def route_after_kt(state: LeaveApplicationState) -> str:
    """
    ROUTER 4: After KT Validator
    
    Decision: Is the KT valid?
    - kt_valid = True  → proceed to notify manager
    - kt_valid = False → loop back (re-show form in production)
    """
    kt_valid = state.get("kt_valid", False)
    if kt_valid:
        print(f"  🔀 ROUTER (kt → notification): KT valid ✅")
        return "kt_valid"
    else:
        print(f"  🔀 ROUTER (kt → retry): KT invalid, need resubmission")
        return "kt_invalid"


def route_after_notification(state: LeaveApplicationState) -> str:
    """
    ROUTER 5: After Notification Agent (after manager responds)
    
    Decision: What did the manager say?
    - 'approved' → commit to database
    - 'rejected' → inform employee, stop
    - 'pending'  → still waiting (shouldn't reach here in normal flow)
    """
    approval = state.get("approval_status", "pending")
    if approval == "approved":
        print(f"  🔀 ROUTER (notification → transaction): manager APPROVED ✅")
        return "approved"
    elif approval == "rejected":
        print(f"  🔀 ROUTER (notification → end): manager REJECTED ❌")
        return "rejected"
    else:
        print(f"  🔀 ROUTER (notification → wait): still pending")
        return "pending"


# ── END NODE FUNCTIONS ─────────────────────────────────────────

def end_ineligible_node(state: LeaveApplicationState) -> dict:
    """Terminal node for ineligible applications."""
    print_node_header("END — INELIGIBLE", "❌")
    reason = state.get("ineligibility_reason", "Eligibility check failed")
    print(f"  ❌ Application rejected: {reason}")
    print(f"  📧 Employee would receive: 'Your leave request could not be processed: {reason}'")
    return {"status": "rejected", "current_node": "end_ineligible"}


def end_rejected_node(state: LeaveApplicationState) -> dict:
    """Terminal node for manager-rejected applications."""
    print_node_header("END — REJECTED BY MANAGER", "❌")
    print(f"  ❌ Manager {state.get('manager_name')} rejected the application")
    print(f"  📧 Employee {state.get('emp_name')} notified of rejection")
    return {"status": "rejected", "current_node": "end_rejected"}


def end_error_node(state: LeaveApplicationState) -> dict:
    """Terminal node for system errors."""
    print_node_header("END — SYSTEM ERROR", "🔴")
    print(f"  🔴 Errors: {state.get('errors', [])}")
    print(f"  📧 Admin notified. Employee notified of technical issue.")
    return {"status": "error", "current_node": "end_error"}


print("✅ All 5 routing functions defined")
print()
print("Conditional edges summary:")
print("  Router 1 (after N1): missing_info? → clarify or proceed")
print("  Router 2 (after N4): is_eligible?  → optimizer or end")
print("  Router 3 (after N5): num_days>=3?  → KT or notify")
print("  Router 4 (after N6): kt_valid?     → notify or retry")
print("  Router 5 (after N7): approved?     → transaction or end")

---
## 🔧 CELL 14 — Graph Assembly

### Theory: Building the StateGraph

Now we wire everything together. The LangGraph API is straightforward:

```python
graph = StateGraph(YourStateClass)   # Create graph with state schema
graph.add_node("name", function)     # Add each node
graph.add_edge("A", "B")             # Add direct edge A → B
graph.add_conditional_edges("A", router, {"key1": "B", "key2": "C"})
graph.set_entry_point("first_node")  # Where execution begins
app = graph.compile()                # Compile into runnable app
```

### Why compile?
Compilation validates the graph (no dangling edges, all nodes connected), optimizes the execution plan, and attaches any checkpointers (like Redis) for state persistence.

In [ ]:
# ============================================================
# CELL 14: Graph Assembly — Wiring Everything Together
# ============================================================

def build_leave_management_graph():
    """
    Builds and compiles the complete LangGraph leave management application.
    
    This function:
    1. Creates a StateGraph with our LeaveApplicationState schema
    2. Registers all 8 processing nodes + 3 terminal nodes
    3. Adds all direct edges (always-proceed connections)
    4. Adds all conditional edges (decision points)
    5. Sets the entry point
    6. Compiles and returns the runnable app
    """
    
    # STEP 1: Create the graph with our state schema
    # LangGraph uses the TypedDict to know what fields exist
    graph = StateGraph(LeaveApplicationState)
    
    # ── STEP 2: REGISTER ALL NODES ────────────────────────────
    # Format: graph.add_node("node_name", function)
    # The name is used in edges and routing functions
    
    # Processing nodes
    graph.add_node("intent_parser",         intent_parser_node)
    graph.add_node("pattern_analyser",      pattern_analyser_node)
    graph.add_node("rag_policy_reader",     rag_policy_reader_node)
    graph.add_node("eligibility_checker",   eligibility_checker_node)
    graph.add_node("leave_optimizer",       leave_optimizer_node)
    graph.add_node("kt_validator",          kt_validator_node)
    graph.add_node("notification_agent",    notification_agent_node)
    graph.add_node("transaction_executor",  transaction_executor_node)
    
    # Terminal nodes (graph ends here based on different outcomes)
    graph.add_node("end_ineligible",        end_ineligible_node)
    graph.add_node("end_rejected",          end_rejected_node)
    graph.add_node("end_error",             end_error_node)
    
    # ── STEP 3: SET ENTRY POINT ───────────────────────────────
    # This is where the graph starts when you call app.invoke()
    graph.set_entry_point("intent_parser")
    
    # ── STEP 4: ADD DIRECT EDGES ──────────────────────────────
    # These edges ALWAYS happen — no conditions
    graph.add_edge("pattern_analyser",     "rag_policy_reader")    # Always read policy after pattern check
    graph.add_edge("rag_policy_reader",    "eligibility_checker")  # Always check eligibility after policy
    
    # Terminal edges (these END nodes go to the actual END)
    graph.add_edge("end_ineligible",       END)
    graph.add_edge("end_rejected",         END)
    graph.add_edge("end_error",            END)
    graph.add_edge("transaction_executor", END)  # Success path ends here
    
    # ── STEP 5: ADD CONDITIONAL EDGES ─────────────────────────
    # Format: add_conditional_edges(source, router_function, {return_value: target_node})
    
    # CONDITIONAL EDGE 1: After Intent Parser
    graph.add_conditional_edges(
        "intent_parser",              # After this node runs...
        route_after_intent_parser,    # ...call this function to decide...
        {                             # ...and go to this node based on return value
            "proceed_to_pattern": "pattern_analyser",  # All info present → continue
            "need_clarification": "intent_parser",     # Missing info → loop back (ask user)
            "end_with_error":     "end_error",         # System error → end
        }
    )
    
    # CONDITIONAL EDGE 2: After Eligibility Checker
    graph.add_conditional_edges(
        "eligibility_checker",
        route_after_eligibility,
        {
            "eligible":   "leave_optimizer",   # Can proceed → generate plans
            "ineligible": "end_ineligible",    # Cannot proceed → show reason, end
        }
    )
    
    # CONDITIONAL EDGE 3: After Leave Optimizer
    graph.add_conditional_edges(
        "leave_optimizer",
        route_after_optimizer,
        {
            "kt_required":       "kt_validator",       # 3+ days → need KT
            "no_kt_needed":      "notification_agent", # < 3 days → skip KT
            "wait_for_selection": "leave_optimizer",   # Guard: loop until selected
        }
    )
    
    # CONDITIONAL EDGE 4: After KT Validator
    graph.add_conditional_edges(
        "kt_validator",
        route_after_kt,
        {
            "kt_valid":   "notification_agent",  # KT complete → notify manager
            "kt_invalid": "kt_validator",        # KT failed → retry form
        }
    )
    
    # CONDITIONAL EDGE 5: After Notification Agent
    graph.add_conditional_edges(
        "notification_agent",
        route_after_notification,
        {
            "approved": "transaction_executor",  # Manager approved → commit to DB
            "rejected": "end_rejected",          # Manager rejected → notify employee
            "pending":  "notification_agent",   # Still waiting → loop (production: Redis handles this)
        }
    )
    
    # ── STEP 6: COMPILE ───────────────────────────────────────
    # compile() validates the graph and returns a runnable
    # In production, add: checkpointer=RedisCheckpointer(redis_url=...)
    app = graph.compile()
    
    return app


# Build the graph
app = build_leave_management_graph()

print("✅ LangGraph compiled successfully!")
print()
print("Graph structure:")
print("  Nodes : intent_parser → pattern_analyser → rag_policy_reader")
print("          → eligibility_checker [BRANCH] → leave_optimizer")
print("          → [BRANCH: KT or not] → kt_validator? → notification_agent")
print("          [BRANCH: approve/reject] → transaction_executor → END")
print()
print("  Conditional edges: 5 decision points")
print("  Direct edges      : 4 always-proceed connections")
print("  Terminal nodes    : 3 (ineligible, rejected, error)")

---
## 🚀 CELL 15 — FULL RUN: Happy Path

### What to Watch For:
- Each node prints its header with timestamp
- Each router prints its decision and reasoning
- The state accumulates data across all 8 nodes
- The final state shows the complete record

**Scenario**: Priya Sharma (EMP-1042) wants 3 days of CL starting next Monday. She is eligible, no patterns, manager approves.

In [ ]:
# ============================================================
# CELL 15: FULL RUN — Happy Path (All Nodes Execute)
# ============================================================

print("🚀 " + "="*58)
print("   STARTING LEAVE MANAGEMENT SYSTEM — HAPPY PATH")
print("   Scenario: Priya Sharma wants 3 days CL from next Monday")
print("   Expected: All 8 nodes run, manager approves, leave saved")
print("="*60)

# Calculate next Monday's date dynamically
today_dt = datetime.now()
days_until_monday = (7 - today_dt.weekday()) % 7
if days_until_monday == 0:
    days_until_monday = 7  # If today is Monday, use next Monday
next_monday = (today_dt + timedelta(days=days_until_monday + 2)).strftime("%Y-%m-%d")
next_wednesday = (today_dt + timedelta(days=days_until_monday + 4)).strftime("%Y-%m-%d")

# ── INITIAL STATE ─────────────────────────────────────────────
# This is what you provide when starting the graph.
# Unspecified fields default to None/[] — LangGraph handles this.
initial_state = {
    "messages": [HumanMessage(content=f"Hi, I'd like to apply for 3 days CL starting {next_monday}. Thanks!")],
    "raw_input": f"Hi, I'd like to apply for 3 days CL starting {next_monday}. Thanks!",
    "current_node": "",
    "errors": [],
    "status": "in_progress",
    
    # Identity (could also be extracted from auth session)
    "emp_id": "EMP-1042",  # Logged-in user's ID
    "emp_name": "",
    "emp_email": "",
    "manager_id": "",
    "manager_email": "",
    "manager_name": "",
    
    # These will be filled by nodes
    "leave_type": "",
    "start_date": "",
    "end_date": "",
    "num_days": 0,
    "reason": "",
    "missing_info": [],
    "clarification_question": "",
    "pattern_flags": [],
    "pattern_risk": "",
    "history_summary": "",
    "policy_text": "",
    "policy_summary": "",
    "policy_rules": {},
    "balance_CL": 0,
    "balance_PL": 0,
    "balance_SL": 0,
    "is_eligible": False,
    "ineligibility_reason": "",
    "leave_plan_options": [],
    "selected_plan": {},
    "plan_selected": False,
    "kt_required": False,
    "kt_receiver_id": "",
    "kt_receiver_name": "",
    "kt_content": "",
    "kt_valid": False,
    "notification_sent": False,
    "approval_status": "pending",
    "email_content": "",
    "final_record_id": "",
    "confirmation_sent": False,
}

print(f"\n📨 INITIAL USER MESSAGE: {initial_state['raw_input']}")
print(f"   Start date calculated: {next_monday}")
print(f"\n{'─'*60}")
print("  EXECUTION BEGINS — Watch each node run in sequence")
print(f"{'─'*60}\n")

# ── RUN THE GRAPH ─────────────────────────────────────────────
# invoke() runs the graph synchronously from start to end
# Returns the final state after all nodes have executed
final_state = app.invoke(initial_state)

# ── FINAL SUMMARY ─────────────────────────────────────────────
print(f"\n{'🎉'*30}")
print("   GRAPH EXECUTION COMPLETE — FINAL SUMMARY")
print(f"{'🎉'*30}")
print(f"  Status          : {final_state.get('status', 'unknown').upper()}")
print(f"  Employee        : {final_state.get('emp_name')} ({final_state.get('emp_id')})")
print(f"  Leave           : {final_state.get('num_days')} days {final_state.get('leave_type')}")
print(f"  Dates           : {final_state.get('start_date')} → {final_state.get('end_date')}")
print(f"  Plan Chosen     : {final_state.get('selected_plan', {}).get('title', 'N/A')}")
print(f"  Pattern Risk    : {final_state.get('pattern_risk', 'N/A')}")
print(f"  KT Done?        : {final_state.get('kt_valid', False)}")
print(f"  Approved by     : {final_state.get('manager_name')}")
print(f"  Record ID       : {final_state.get('final_record_id', 'Not saved')}")
print(f"  Nodes executed  : N1 → N2 → N3 → N4 → N5 → N6 → N7 → N8")
print(f"{'─'*60}")

In [ ]:
# ============================================================
# CELL 16: EDGE CASE — Ineligible Employee (Zero Balance)
# ============================================================
# Scenario: EMP-9999 (probation, zero balance) tries to apply for CL
# Expected: N1 → N2 → N3 → N4 (FAILS) → end_ineligible

print("🔴 " + "="*58)
print("   EDGE CASE: INELIGIBLE EMPLOYEE")
print("   Employee EMP-9999 on probation, requesting CL")
print("   Expected: N4 Eligibility fails → graph ends early")
print("="*60)

next_monday2 = (datetime.now() + timedelta(days=8)).strftime("%Y-%m-%d")

ineligible_state = {
    **initial_state,  # Use same template
    "messages": [HumanMessage(content=f"I want 2 days CL from {next_monday2}")],
    "raw_input": f"I want 2 days CL from {next_monday2}",
    "emp_id": "EMP-9999",  # Probation employee with zero balance
    "status": "in_progress",
}

print(f"\n📨 Input: '{ineligible_state['raw_input']}' from EMP-9999 (on probation)\n")

result_ineligible = app.invoke(ineligible_state)

print(f"\n{'─'*60}")
print(f"  RESULT: {result_ineligible.get('status', '').upper()}")
print(f"  Reason: {result_ineligible.get('ineligibility_reason', 'N/A')}")
print(f"  Note  : Graph exited at N4 without running N5-N8 (as designed)")
print(f"{'─'*60}")

In [ ]:
# ============================================================
# CELL 17: EDGE CASE — Short Leave (No KT Required)
# ============================================================
# Scenario: Priya takes just 1 day SL
# Expected: N1 → N2 → N3 → N4 → N5 → (skip N6) → N7 → N8

print("🔵 " + "="*58)
print("   EDGE CASE: SHORT LEAVE (< 3 days, KT skipped)")
print("   EMP-1042 takes 1 day SL tomorrow")
print("   Expected: KT Validator SKIPPED, goes straight to notification")
print("="*60)

tomorrow = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")

short_leave_state = {
    **initial_state,
    "messages": [HumanMessage(content=f"I'm unwell, need 1 day SL tomorrow ({tomorrow})")],
    "raw_input": f"I'm unwell, need 1 day SL tomorrow ({tomorrow})",
    "emp_id": "EMP-1042",
    "status": "in_progress",
}

print(f"\n📨 Input: '{short_leave_state['raw_input']}'\n")

result_short = app.invoke(short_leave_state)

print(f"\n{'─'*60}")
print(f"  RESULT         : {result_short.get('status', '').upper()}")
print(f"  KT Required    : {result_short.get('kt_required', False)} (expected: False ✅)")
print(f"  KT Validator   : SKIPPED (as designed for < 3 days)")
print(f"  Record ID      : {result_short.get('final_record_id', 'N/A')}")
print(f"{'─'*60}")

In [ ]:
# ============================================================
# CELL 18: EDGE CASE — Pattern Detected (High Risk)
# ============================================================
# Scenario: EMP-9999 had a monday pattern, now applying again on Monday
# But wait - EMP-9999 is ineligible (probation). 
# Let's temporarily add history for EMP-1042 to demonstrate pattern detection

print("⚠️  " + "="*57)
print("   EDGE CASE: PATTERN DETECTION DEMO")
print("   Temporarily adding Monday pattern to EMP-1042's history")
print("   Expected: Pattern flagged, but application still proceeds")
print("="*60)

# Temporarily add suspicious pattern to EMP-1042's history
original_history = LEAVE_HISTORY_DB.get("EMP-1042", [])
LEAVE_HISTORY_DB["EMP-1042"] = [
    {"date": (today_dt - timedelta(days=65)).strftime("%Y-%m-%d"), "day": "Monday", "type": "CL"},
    {"date": (today_dt - timedelta(days=49)).strftime("%Y-%m-%d"), "day": "Monday", "type": "CL"},
    {"date": (today_dt - timedelta(days=28)).strftime("%Y-%m-%d"), "day": "Monday", "type": "CL"},
]

next_monday3 = (datetime.now() + timedelta(days=8)).strftime("%Y-%m-%d")

pattern_state = {
    **initial_state,
    "messages": [HumanMessage(content=f"Want to take 2 days CL from {next_monday3}")],
    "raw_input": f"Want to take 2 days CL from {next_monday3}",
    "emp_id": "EMP-1042",
    "status": "in_progress",
}

print(f"\n📨 Input: '{pattern_state['raw_input']}'")
print(f"   EMP-1042 now has 3 Monday leaves in history — should flag\n")

result_pattern = app.invoke(pattern_state)

# Restore original history
LEAVE_HISTORY_DB["EMP-1042"] = original_history

print(f"\n{'─'*60}")
print(f"  RESULT      : {result_pattern.get('status', '').upper()}")
print(f"  Pattern Flags: {result_pattern.get('pattern_flags', [])}")  
print(f"  Pattern Risk : {result_pattern.get('pattern_risk', 'N/A')}")
print(f"  Key Point    : Application still PROCEEDED — pattern only flags, never blocks")
print(f"  Manager was  : Notified with pattern warning in email")
print(f"  History      : Restored to original ✅")
print(f"{'─'*60}")

In [ ]:
# ============================================================
# CELL 19: Graph Visualization
# ============================================================

print("📊 GRAPH VISUALIZATION")
print()

# Try to get the Mermaid diagram from LangGraph
try:
    mermaid_code = app.get_graph().draw_mermaid()
    print("Mermaid diagram code (paste at https://mermaid.live to visualize):")
    print()
    print(mermaid_code)
except Exception as e:
    print(f"Note: {e}")
    print()
    # Manual ASCII diagram as fallback
    print("GRAPH STRUCTURE (ASCII):")
    print("""
  ┌─────────────────┐
  │  [START]        │
  └────────┬────────┘
           │
           ▼
  ┌─────────────────┐     missing_info?    ┌─────────────────┐
  │  N1: Intent     │ ──────────────────► │  (ask user &    │
  │     Parser      │ ◄───────────────── │   loop back)    │
  └────────┬────────┘   all fields filled  └─────────────────┘
           │
           ▼
  ┌─────────────────┐
  │  N2: Pattern    │  (always proceeds, never blocks)
  │    Analyser     │
  └────────┬────────┘
           │
           ▼
  ┌─────────────────┐
  │  N3: RAG Policy │  (reads policy from storage)
  │     Reader      │
  └────────┬────────┘
           │
           ▼
  ┌─────────────────┐  ineligible  ┌─────────────────┐
  │  N4: Eligibility│ ───────────► │  END: Ineligible │
  │     Checker     │              └─────────────────┘
  └────────┬────────┘
           │ eligible
           ▼
  ┌─────────────────┐
  │  N5: Leave Plan │  ◄─── HITL pause: user picks a card
  │    Optimizer ⭐ │
  └────────┬────────┘
           │
     ┌─────┴─────┐
     │           │
  days≥3?    days<3?
     │           │
     ▼           │
  ┌──────────┐  │
  │  N6: KT  │  │
  │ Validator│  │
  └────┬─────┘  │
       │        │
       └───┬────┘
           │
           ▼
  ┌─────────────────┐
  │  N7: Notification│ ◄─── HITL pause: Redis thread waits for manager
  │      Agent      │
  └────────┬────────┘
           │
     ┌─────┴─────┐
     │           │
  approved?  rejected?
     │           │
     ▼           ▼
  ┌──────────┐  ┌─────────────────┐
  │  N8: TX  │  │  END: Rejected  │
  │ Executor │  └─────────────────┘
  └────┬─────┘
       │
       ▼
  ┌─────────────────┐
  │  [END: SUCCESS] │
  └─────────────────┘
    """)

---
## 🏭 CELL 20 — Production Checklist & What to Build Next

This cell documents everything you need to do to take this from notebook to production.

In [ ]:
# ============================================================
# CELL 20: Production Checklist & Next Steps
# ============================================================

checklist = """
╔══════════════════════════════════════════════════════════════╗
║          PRODUCTION READINESS CHECKLIST                     ║
╚══════════════════════════════════════════════════════════════╝

── REPLACE MOCK DATA WITH REAL INTEGRATIONS ──────────────────

  [ ] Employee DB     → Connect to HRMS API (Oracle HCM / SAP SuccessFactors)
  [ ] Leave Balance   → HRMS leave balance endpoint
  [ ] Leave History   → SQL query on leave_records table
  [ ] Policy Store    → Embed PDFs into Oracle DB 23ai / Pinecone
                        Use langchain OracleVectorStore or PineconeVectorStore
  [ ] Calendar        → Microsoft Graph API or Google Calendar API
  [ ] Email Sending   → SendGrid / AWS SES (replace print with real send)
  [ ] Audit Trail     → Write every state change to an audit log table

── STATE PERSISTENCE (Redis HITL) ────────────────────────────

  [ ] Install: pip install langgraph-checkpoint-redis
  [ ] Setup Redis (AWS ElastiCache / Azure Cache / Redis Cloud)
  [ ] Replace: graph.compile() with:
               from langgraph.checkpoint.redis import RedisSaver
               checkpointer = RedisSaver(redis_client)
               graph.compile(checkpointer=checkpointer)
  [ ] Generate thread_id per application session
  [ ] Implement /api/leave/approve webhook
  [ ] Implement /api/leave/reject webhook
  [ ] Set approval token expiry (72 hours)
  [ ] Handle timeout (re-send email or auto-escalate)

── SECURITY ──────────────────────────────────────────────────

  [ ] Move OPENAI_API_KEY to .env file + python-dotenv
  [ ] NEVER commit API keys to git (.gitignore .env)
  [ ] Use HMAC-signed tokens in approval URLs (not just UUID)
  [ ] One-time use tokens (mark as used after click)
  [ ] Rate limit the approval webhook endpoint
  [ ] Validate emp_id from authenticated session, not user input

── PRODUCTION LangGraph PATTERNS ─────────────────────────────

  [ ] Use thread_id per user session: {"configurable": {"thread_id": "user-123-session-456"}}
  [ ] For HITL pause: use interrupt_before=["notification_agent"] or NodeInterrupt
  [ ] For streaming: use app.astream() for async/FastAPI
  [ ] For long-running: use background workers (Celery) with Redis
  [ ] Add retry logic: tenacity decorators on LLM calls
  [ ] Add fallback: if LLM JSON parse fails, retry with temperature=0

── OBSERVABILITY ─────────────────────────────────────────────

  [ ] LangSmith tracing (Anthropic's hosted tracing for LangChain)
       os.environ["LANGCHAIN_TRACING_V2"] = "true"
       os.environ["LANGCHAIN_API_KEY"] = "your-langsmith-key"
  [ ] Structured logging (Python logging module → CloudWatch / Datadog)
  [ ] Track token usage per node (cost monitoring)
  [ ] Alert on error rate > threshold

── TESTING ───────────────────────────────────────────────────

  [ ] Unit tests per node (mock LLM with LangChain FakeListLLM)
  [ ] Integration tests for each routing path
  [ ] End-to-end tests: happy path + all edge cases
  [ ] Load test: concurrent applications (Redis checkpointer handles this)

── WHAT TO BUILD NEXT ────────────────────────────────────────

  [ ] FastAPI wrapper: POST /api/apply-leave → starts graph
  [ ] React frontend: chat interface + plan card UI
  [ ] LangSmith dashboard: monitor all agent runs
  [ ] Expand pattern analyser to 12-month history
  [ ] Add half-day leave support (num_days as float)
  [ ] Multi-level approval (manager + HR for PL > 5 days)
  [ ] Leave cancellation flow (separate graph)
  [ ] Annual leave report agent (separate graph)

── KEY LANGGRAPH CONCEPTS TO MASTER NEXT ─────────────────────

  [ ] Subgraphs: embed one StateGraph inside another
  [ ] Parallel nodes: run N2 and N3 in parallel (map-reduce)
  [ ] Streaming: app.astream_events() for real-time token streaming
  [ ] Time travel: replay from any checkpoint (great for debugging)
  [ ] LangGraph Platform: deploy graphs as managed microservices

╔══════════════════════════════════════════════════════════════╗
║  QUICK REFERENCE — Key LangGraph API                        ║
╠══════════════════════════════════════════════════════════════╣
║  StateGraph(State)                    # Create graph        ║
║  graph.add_node("name", fn)           # Add node           ║
║  graph.add_edge("A", "B")             # Direct edge        ║
║  graph.add_conditional_edges(         # Conditional edge    ║
║      "A", router_fn, {"x": "B"})                           ║
║  graph.set_entry_point("node")        # Start here         ║
║  app = graph.compile(checkpointer=X)  # Build it           ║
║  app.invoke(state)                    # Run it             ║
║  app.stream(state)                    # Stream it          ║
║  app.get_graph().draw_mermaid()       # Visualize it       ║
╚══════════════════════════════════════════════════════════════╝
"""

print(checklist)
print()
print("🎯 You now have a complete, runnable, production-reference LangGraph project.")
print("   Every node is explained. Every edge is documented. Every output is visible.")
print("   Good luck at your MNC! 🚀")